In [ ]:
#| default_exp eod_download

In [ ]:
#| export
from pathlib import Path

In [ ]:
import panel as pn
import pandas as pd
import datetime as dt

from fastcore.all import patch

import eodhdapi

In [ ]:
pn.extension('tabulator')

In [ ]:
#| export
EOD_DIR = Path.home()/'.cache'/'eod_data'
EOD_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
#| export
from portfolio.util import data_path

## Base

In [ ]:
class AssetBase():
    def __init__(self, save_dir=EOD_DIR):
        self.save_dir = save_dir
        self.date_range = pn.widgets.DateRangePicker(name='Date Range', value=(dt.date(2015,1,1), dt.date.today()))

In [ ]:
@patch
def _search(self: AssetBase, isin: str) -> pd.DataFrame:
    return pd.DataFrame(eodhdapi.eodhd_search(isin))[['Name','Code','Exchange','Type','Currency','ISIN']]

In [ ]:
@patch
def _download(self: AssetBase, code: str, exchange: str) -> pd.DataFrame:
    start, end = self.date_range.value
    return eodhdapi.get_stock_data(f"{code}.{exchange}", period='d', from_date=str(start), to_date=str(end))

In [ ]:
@patch
def _save(self: AssetBase, isin: str, data: pd.DataFrame) -> Path:
    path = self.save_dir / f"{isin}.csv"
    data.to_csv(path)
    return path

In [ ]:
@patch
def _search_and_display(self: AssetBase, isin: str, tbl, status):
    try:
        df = self._search(isin)
        tbl.value = df
        status.object = f'{len(df)} result(s) found.'
        return df
    except Exception as ex:
        status.object = f'❌ {ex}'
        return None

In [ ]:
@patch
def _download_row(self: AssetBase, row, status, preview=None):
    try:
        df = self._download(row.Code, row.Exchange)[['adjusted_close']]
        df = df.assign(**row)
        if preview is not None:
            preview.value = df.tail(12)
            status.object = f'Preview ready ({len(df)} rows). Confirm to save.'
            return df
        name = row.ISIN if row.ISIN else f'{row.Code}_{row.Exchange}'
        path = self._save(name, df)
        status.object = f'✅ Saved {name} ({len(df)} rows) to `{path}`'
        return df
    except Exception as ex:
        status.object = f'❌ {ex}'
        return None

## Single

In [ ]:
class AssetDownloader(AssetBase):
    def __init__(self, save_dir=EOD_DIR):
        super().__init__(save_dir=save_dir)
        self.search_input = pn.widgets.TextInput(name='ISIN', placeholder='e.g. IE00BDFL4P12', width=250)
        self.results_tbl = pn.widgets.Tabulator(pd.DataFrame(), selectable='checkbox-single', show_index=False, height=180)
        self.status = pn.pane.Markdown('')
        self.state = {}
        self.preview = pn.widgets.Tabulator(pd.DataFrame(), show_index=True, height=220)
        self.search_btn = pn.widgets.Button(name='Search', button_type='primary')
        self.save_btn = pn.widgets.Button(name='Save', button_type='success', disabled=True)
        self.dl_btn = pn.widgets.Button(name='Download Preview', disabled=True)
        self._bound = False

In [ ]:
@patch
def on_search(self: AssetDownloader, e):
    self.results_tbl.selection = []
    i = self.search_input.value.strip()
    if not i: return
    self.status.object = f'Searching `{i}`…'
    df = self._search_and_display(i, self.results_tbl, self.status)
    if df is not None: self.dl_btn.disabled = False

In [ ]:
@patch
def on_download(self: AssetDownloader, e):
    sel = self.results_tbl.selection
    if not sel: self.status.object = '⚠️ Select a row first.'; return
    row = self.results_tbl.value.iloc[sel[0]]
    self.status.object = f'Downloading {row.Code}.{row.Exchange}…'
    df = self._download_row(row, self.status, self.preview)
    if df is not None:
        self.state.update(data=df, isin=row.ISIN if row.ISIN else f'{row.Code}_{row.Exchange}')
        self.save_btn.disabled = False

In [ ]:
@patch
def on_save(self: AssetDownloader, e):
    path = self._save(self.state['isin'], self.state['data'])
    self.save_btn.disabled = True
    self.status.object = f'✅ Saved to `{path}`'

In [ ]:
@patch
def _bind(self: AssetDownloader):
    if not self._bound:
        self.search_btn.on_click(self.on_search)
        self.dl_btn.on_click(self.on_download)
        self.save_btn.on_click(self.on_save)
        self._bound = True

In [ ]:
@patch
def layout(self: AssetDownloader):
    self._bind()
    return pn.Column(
        '## Asset Downloader',
        pn.Row(self.search_input, self.search_btn),
        self.status, self.results_tbl,
        self.date_range,
        pn.Row(self.dl_btn, self.save_btn),
        '### Preview', self.preview,
    )

## Batch

In [ ]:
class BatchAssetDownloader(AssetBase):
    def __init__(self, save_dir=EOD_DIR):
        super().__init__(save_dir=save_dir)
        self.file_input = pn.widgets.TextInput(name='Transactions CSV', value=f'{data_path()}/sample_transactions_test.csv', width=300)
        self.search_input = pn.widgets.TextInput(name='ISIN', width=250)
        self.results_tbl = pn.widgets.Tabulator(pd.DataFrame(), selectable='checkbox-single', show_index=False, height=180)
        self.status = pn.pane.Markdown('')
        self.load_btn = pn.widgets.Button(name='Load ISINs', button_type='primary')
        self.next_btn = pn.widgets.Button(name='Confirm & Next', button_type='primary')
        self.dl_btn = pn.widgets.Button(name='Download All', button_type='success', disabled=True)
        self.isins = []
        self.idx = 0
        self.confirmed = {}
        self._bound = False

In [ ]:
@patch
def on_load(self: BatchAssetDownloader, e):
    path = self.file_input.value.strip()
    if not path: self.status.object = '⚠️ Enter a file path.'; return
    try:
        df = pd.read_csv(path)
        self.isins = list(df.ISIN.unique())
        self.idx = 0
        self.confirmed = {}
        self._show_current()
    except Exception as ex: self.status.object = f'❌ {ex}'

In [ ]:
@patch
def _show_current(self: BatchAssetDownloader):
    if self.idx >= len(self.isins):
        self.status.object = '✅ All ISINs confirmed! Click Download All.'
        self.dl_btn.disabled = False
        self.next_btn.disabled = True
        self.results_tbl.value = pd.DataFrame()
        return
    isin = self.isins[self.idx]
    self.status.object = f'**ISIN {self.idx+1} of {len(self.isins)}**: `{isin}`'
    self.search_input.value = isin
    self._search_and_display(isin, self.results_tbl, self.status)

In [ ]:
@patch
def on_next(self: BatchAssetDownloader, e):
    sel = self.results_tbl.selection
    if not sel: self.status.object = '⚠️ Select a row first.'; return
    row = self.results_tbl.value.iloc[sel[0]]
    self.confirmed[self.isins[self.idx]] = row
    self.idx += 1
    self.results_tbl.selection = []
    self._show_current()

In [ ]:
@patch
def on_download(self: BatchAssetDownloader, e):
    for isin, row in self.confirmed.items():
        self.status.object = f'Downloading {row.Code}.{row.Exchange}…'
        df = self._download_row(row, self.status)
        if df is None: return
    self.status.object = f'✅ Saved {len(self.confirmed)} files to `{self.save_dir}`'

In [ ]:
@patch
def _bind(self: BatchAssetDownloader):
    if not self._bound:
        self.load_btn.on_click(self.on_load)
        self.next_btn.on_click(self.on_next)
        self.dl_btn.on_click(self.on_download)
        self._bound = True

In [ ]:
@patch
def layout(self: BatchAssetDownloader):
    self._bind()
    return pn.Column(
        '## Batch Asset Downloader',
        pn.Row(self.file_input, self.load_btn),
        self.status,
        self.results_tbl,
        self.date_range,
        pn.Row(self.next_btn, self.dl_btn),
    )

## Update prices

In [ ]:
class AssetUpdater(AssetBase):
    def __init__(self, save_dir=EOD_DIR):
        super().__init__(save_dir=save_dir)
        self.update_btn = pn.widgets.Button(name='Update All', button_type='primary')
        self.status = pn.pane.Markdown('')
        self.progress = pn.widgets.Progress(name='Progress', value=0, max=1)
        self._bound = False

In [ ]:
@patch
def on_update(self: AssetUpdater, e):
    files = sorted(self.save_dir.glob('*.csv'))
    if not files:
        self.status.object = '⚠️ No CSV files found.'
        return
    self.progress.max = len(files)
    self.progress.value = 0
    updated = 0
    for i, path in enumerate(files):
        self.progress.value = i
        df = pd.read_csv(path, parse_dates=['date'], index_col='date')
        last_date = df.index.max()
        code = df['Code'].iloc[0]
        exchange = df['Exchange'].iloc[0]
        self.status.object = f'Updating `{path.stem}` ({i+1}/{len(files)})… fetching from {last_date.date()}'
        try:
            new = self._download(code, exchange)[['adjusted_close']]
            meta = df.iloc[0].drop('adjusted_close')
            new = new.assign(**meta)
            if len(new) > 0:
                df = pd.concat([df, new])
                df = df[~df.index.duplicated(keep='last')]
                df.to_csv(path)
                updated += 1
        except Exception as ex:
            self.status.object = f'❌ {path.stem}: {ex}'
            return
    self.progress.value = len(files)
    self.status.object = f'✅ Updated {updated}/{len(files)} files.'

In [ ]:
@patch
def _bind(self: AssetUpdater):
    if not self._bound:
        self.update_btn.on_click(self.on_update)
        self._bound = True

In [ ]:
@patch
def layout(self: AssetUpdater):
    self._bind()
    return pn.Column(
        '## Update Prices',
        self.update_btn,
        self.progress,
        self.status,
    )

## App

In [ ]:
# aw = AssetDownloader()
# bw = BatchAssetDownloader()
# uw = AssetUpdater()

In [ ]:
#app = lambda: pn.panel(pn.Tabs(('Single', aw.layout()), ('Batch', bw.layout()), ('Update', uw.layout())))
#server = pn.serve(app, port=6004, show=True, allow_websocket_origin=['vigilant-spiral-dances-oycg2n.solveit.pub'])

In [ ]:
#server.stop()